# Module 21 — Fault-Tolerant Agent Coordination

Executable lab for message envelopes, leases, heartbeats, retries, idempotency, dead letters, backpressure and cancellation.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
import time, uuid, random


In [ ]:
class Status(str,Enum): PENDING='pending'; LEASED='leased'; COMPLETED='completed'; RECOVERABLE='recoverable'; CANCELLED='cancelled'; DEAD='dead_letter'

@dataclass
class Task:
    id:str; tenant:str; status:Status=Status.PENDING; lease:str|None=None; until:float|None=None; attempts:int=0; history:list=field(default_factory=list)

class Coordinator:
    def __init__(self,lease_seconds=5,max_attempts=3): self.tasks={}; self.effects={}; self.lease_seconds=lease_seconds; self.max_attempts=max_attempts
    def create(self,tenant):
        t=Task(str(uuid.uuid4()),tenant); self.tasks[t.id]=t; return t
    def claim(self,t,worker):
        now=time.monotonic()
        if t.status==Status.CANCELLED: raise RuntimeError('cancelled')
        if t.status==Status.LEASED and t.until and t.until>now: raise RuntimeError('already leased')
        t.status=Status.LEASED; t.lease=f'{worker}:{uuid.uuid4()}'; t.until=now+self.lease_seconds; t.attempts+=1; t.history.append('claimed'); return t.lease
    def heartbeat(self,t,lease):
        if t.status!=Status.LEASED or t.lease!=lease: raise RuntimeError('stale lease')
        t.until=time.monotonic()+self.lease_seconds
    def complete(self,t,lease,result):
        if t.status!=Status.LEASED or t.lease!=lease: raise RuntimeError('stale worker')
        t.status=Status.COMPLETED; t.history.append('completed'); return result
    def expire(self):
        now=time.monotonic()
        for t in self.tasks.values():
            if t.status==Status.LEASED and t.until<=now: t.status=Status.RECOVERABLE; t.history.append('expired')
    def effect(self,key,result):
        if key in self.effects: return False
        self.effects[key]=result; return True


## 1. Lease ownership
Claim a task, then attempt a competing claim. The second worker must be rejected while the lease is valid.

In [ ]:
c=Coordinator(lease_seconds=5); t=c.create('tenant-a'); lease=c.claim(t,'worker-1'); print(t.status,lease)
try: c.claim(t,'worker-2')
except RuntimeError as e: print('DENIED:',e)


## 2. Heartbeat and stale worker
Heartbeat extends the current lease. A stale lease cannot heartbeat or commit.

In [ ]:
c.heartbeat(t,lease); c.complete(t,lease,'evidence'); print(t.status,t.history)
try: c.complete(t,lease,'duplicate')
except RuntimeError as e: print('STALE COMMIT BLOCKED:',e)


## 3. Idempotency
At-least-once delivery is safe only when side effects are deduplicated.

In [ ]:
print(c.effect('payment-123','completed')); print(c.effect('payment-123','completed-again'))


## 4. Lease expiry and recovery
Create an immediately expiring lease. Recover it, then allow another worker to claim it. Extend this with durable state in the Module 1 store.

In [ ]:
r=Coordinator(lease_seconds=0); x=r.create('tenant-a'); r.claim(x,'worker-1'); time.sleep(.01); r.expire(); print(x.status,x.history); new_lease=r.claim(x,'worker-2'); print('reclaimed:',new_lease)


## 5. Retry taxonomy
Exercise: retry transient/timeout failures, but dead-letter authorization and permanent failures. Add exponential backoff with jitter and a global deadline.

In [ ]:
def backoff(attempt,base=.5,cap=30): return min(cap,base*2**max(0,attempt-1))*random.uniform(.8,1.2)
for i in range(1,5): print(i,round(backoff(i),3))


## 6. Backpressure
Simulate queue overload. Exercise: reject or defer new work when queue depth or budget crosses a threshold rather than allowing an agent swarm to amplify load.

In [ ]:
queue_depth=120; limit=100
print('ADMIT' if queue_depth<limit else 'BACKPRESSURE: reject/defer')


## 7. Failure injection matrix
Inject duplicate messages, out-of-order events, stale workers, coordinator crashes, queue floods, partitions, poisoned context and worker disagreement. For every failure document detection, containment and regression test.

# Exercises
1. Add versioned message envelopes.
2. Add correlation/causation IDs.
3. Persist leases.
4. Reject stale commits after lease reassignment.
5. Add heartbeats and progress markers.
6. Implement retry classification.
7. Add a dead-letter queue.
8. Implement bounded admission control.
9. Propagate cancellation to children.
10. Implement quorum with correlated-failure tests.
11. Build distributed trace reconstruction.
12. Integrate Module 18 security and Module 20 worker contracts.

# Gold challenge
Build AegisAI Fault-Tolerant Agent Network with durable messages, leases, heartbeats, idempotency, bounded retries, dead letters, backpressure, cancellation, tenant/capability controls and complete distributed run reconstruction.